# Computer Networks · Week 1 — Applications notebook (Google Colab)
### Introduction: what a network is and how the Internet is put together

Companion notebook to the **Week 1 teaching pack** (Lecture 1, Seminar 1, Laboratory 1 Linux, Laboratory 1 Windows).
Every part of the pack has a matching part here, which you can run:

| Part | In the teaching pack | In this notebook |
|---|---|---|
| **A** | Lecture 1 | Calculators and simulations for every formula and worked example; the end-of-lecture exercises checked by code |
| **B** | Seminar 1: reasoning with numbers | Problems 1–8 solved in Python, plus a queue simulator, a working *time protocol* (client + server) and a traceroute analyser |
| **C** | Laboratory 1 (Linux) | Tasks 1–8 run on the Colab virtual machine, which is a real Linux host in a Google data centre |
| **D** | Laboratory 1 (Windows) | The same tasks with the Windows/PowerShell way of thinking, using Python (and, optionally, real PowerShell 7) |

**How to use it.** Run the cells in order (`Runtime → Run all` also works). Cells marked ✍️ are for your answers to the questions Q1–Q16, so that this notebook can be handed in as your report (`File → Download → .ipynb`).

> **About the Colab machine.** Colab gives you a Linux VM in a Google data centre, not a machine in the faculty laboratory. Your addresses, gateway and paths will be those of that data centre. Some cloud networks block ICMP (the packets used by `ping` and some `traceroute` probes). Where that happens the notebook falls back to TCP-based measurements and tells you why.

## 0 · Setup

In [ ]:
# ---------------------------------------------------------------
#  Helper functions used throughout the notebook. Run this cell first.
# ---------------------------------------------------------------
import os, re, sys, time, math, json, socket, shutil, subprocess, threading, ipaddress
from collections import deque, Counter

IN_COLAB = 'google.colab' in sys.modules
SUDO = '' if os.geteuid() == 0 else 'sudo '

def sh(cmd, timeout=60, show=True):
    """Run a shell command, print its output (stdout+stderr), never raise."""
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        out = (r.stdout or '') + (r.stderr or '')
    except subprocess.TimeoutExpired as e:
        o = e.stdout or ''
        out = (o.decode(errors='replace') if isinstance(o, bytes) else o) + f'\n[stopped after {timeout} s]'
    if show:
        print(f'$ {cmd}\n{out.rstrip()}\n')
    return out

def have(tool):
    return shutil.which(tool) is not None

def default_route():
    """(interface, gateway) used to reach the Internet."""
    out = sh('ip route get 1.1.1.1', show=False)
    dev = re.search(r'\bdev (\S+)', out)
    via = re.search(r'\bvia (\S+)', out)
    src = re.search(r'\bsrc (\S+)', out)
    return (dev.group(1) if dev else 'any', via.group(1) if via else None, src.group(1) if src else None)

def resolve(host):
    try:
        return socket.gethostbyname(host)
    except Exception as e:
        print(f'cannot resolve {host}: {e}')
        return None

def tcp_rtt(host, port=443, n=5, timeout=3):
    """Measure RTT as the time a TCP connect() takes (one SYN / SYN-ACK exchange).
    Works where ICMP ping is blocked (as on many cloud VMs)."""
    ip = resolve(host)
    if not ip:
        return []
    rtts = []
    for _ in range(n):
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(timeout)
        t0 = time.perf_counter()
        try:
            s.connect((ip, port))
            rtts.append((time.perf_counter() - t0) * 1000)
        except Exception as e:
            print(f'  connect to {ip}:{port} failed: {e}')
        finally:
            s.close()
        time.sleep(0.2)
    if rtts:
        print(f'TCP-connect RTT to {host} ({ip}:{port}): min/avg/max = '
              f'{min(rtts):.2f}/{sum(rtts)/len(rtts):.2f}/{max(rtts):.2f} ms  ({len(rtts)}/{n} answered)')
    return rtts

def fmt_time(seconds):
    """Pretty-print a duration."""
    s = abs(seconds)
    if s < 1e-6: return f'{seconds*1e9:.3g} ns'
    if s < 1e-3: return f'{seconds*1e6:.4g} µs'
    if s < 1:    return f'{seconds*1e3:.4g} ms'
    if s < 120:  return f'{seconds:.4g} s'
    if s < 7200: return f'{seconds/60:.4g} min'
    if s < 2*86400: return f'{seconds/3600:.4g} h'
    return f'{seconds/86400:.4g} days'

def check(name, got, expected, rel=0.02):
    ok = math.isclose(got, expected, rel_tol=rel, abs_tol=1e-12)
    print(f"{'✅' if ok else '❌'} {name}: computed {got:.6g}, expected ≈ {expected:.6g}")
    return ok

print('Helpers loaded. Running in Colab:', IN_COLAB, '| root:', os.geteuid() == 0)

In [ ]:
# Install the command-line tools used in Part C (about 30 s on Colab).
pkgs = 'iproute2 iputils-ping traceroute mtr-tiny dnsutils curl netcat-openbsd tcpdump'
if os.geteuid() == 0 or have('sudo'):
    sh(f'{SUDO}apt-get -qq update > /dev/null 2>&1; DEBIAN_FRONTEND=noninteractive {SUDO}apt-get -qq install -y {pkgs} > /dev/null 2>&1; echo done', timeout=600)
for t in ['ip', 'ping', 'traceroute', 'mtr', 'dig', 'curl', 'nc', 'ss', 'tcpdump']:
    print(f'{t:11s}', '✅' if have(t) else '❌ missing')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize': (9, 4.5), 'axes.grid': True, 'grid.alpha': .3})

---
# Part A · Lecture 1 — the lecture in code

## A.1 Units: always convert to bits first
Rates are in **bits per second** with **decimal** prefixes (1 Mbit/s = 10⁶ bit/s); file sizes are in **bytes** (1 MB = 10⁶ B unless stated). Signals in copper and fibre travel at about **2 × 10⁸ m/s = 200 km/ms**.

In [ ]:
K, M, G, T = 1e3, 1e6, 1e9, 1e12
BYTE = 8                    # bits
S_FIBRE = 2e8               # m/s, copper and fibre
S_AIR = 3e8                 # m/s, radio and vacuum

def bits(nbytes):           # bytes -> bits
    return nbytes * BYTE

def t_trans(L_bits, R_bps):  # transmission delay L/R
    return L_bits / R_bps

def t_prop(d_m, s=S_FIBRE):  # propagation delay d/s
    return d_m / s

print('1500-byte packet =', bits(1500), 'bits')
print('1 MiB / 1 MB =', 2**20 / 1e6, '→ binary units are ~4.9 % larger')

## A.2 Packet switching: store-and-forward
A switch must receive a whole packet before forwarding it. One packet over **N** links of rate **R**: `d = N·L/R`. **P** packets sent back to back: `d = (N + P − 1)·L/R`, because the links work in parallel on different packets (a *pipeline*).

In [ ]:
def store_and_forward(L_bits, R_bps, N_links, P_packets=1):
    return (N_links + P_packets - 1) * L_bits / R_bps

# Worked example 1 — 1500-byte packet, 100 Mbit/s links
L = bits(1500)
print('L/R on one link      :', fmt_time(t_trans(L, 100*M)))
print('two links, one router:', fmt_time(store_and_forward(L, 100*M, 2)))
check('Worked example 1', store_and_forward(L, 100*M, 2), 0.24e-3)

In [ ]:
def pipeline_chart(L_bits, R_bps, N_links, P_packets, title=''):
    """Gantt chart: when each packet occupies each link (store-and-forward, no propagation)."""
    tt = L_bits / R_bps
    fig, ax = plt.subplots(figsize=(9, 0.6 * N_links + 1.5))
    colors = plt.cm.tab10.colors
    for p in range(P_packets):
        for n in range(N_links):
            start = (p + n) * tt
            ax.barh(n, tt * 1e3, left=start * 1e3, color=colors[p % 10], edgecolor='k')
            ax.text((start + tt / 2) * 1e3, n, f'P{p+1}', ha='center', va='center', fontsize=8, color='w')
    ax.set_yticks(range(N_links), [f'link {n+1}' for n in range(N_links)])
    ax.invert_yaxis(); ax.set_xlabel('time (ms)')
    total = store_and_forward(L_bits, R_bps, N_links, P_packets)
    ax.set_title(title or f'{P_packets} packets over {N_links} links: last bit arrives at {total*1e3:.3g} ms')
    plt.tight_layout(); plt.show()

# Lecture exercise 1: five 1500-byte packets, three 10 Mbit/s links
pipeline_chart(bits(1500), 10*M, 3, 5)

## A.3 Circuit switching (TDM)
With time-division multiplexing each circuit gets one slot of every frame — `R/slots` — whether the other slots are used or not.

In [ ]:
def tdm_transfer(file_bits, R_bps, slots, setup_s=0.0):
    per_circuit = R_bps / slots
    return per_circuit, setup_s + file_bits / per_circuit

rate, total = tdm_transfer(640_000, 2.048*M, 32, setup_s=0.5)
print(f'Each circuit gets {rate/1e3:.0f} kbit/s; the file takes {total:.1f} s (including 0.5 s set-up)')
check('Worked example 2', total, 10.5)

## A.4 Packet vs circuit switching: statistical multiplexing
A 1 Gbit/s link; each user sends at 100 Mbit/s when active and is active 10 % of the time.
Circuit switching admits 10 users. With packet switching, demand exceeds capacity only when **11 or more** users are active at once. The number of active users is binomial:

$$P(X \ge 11) = \sum_{k=11}^{N} \binom{N}{k}\,p^k (1-p)^{N-k}$$

In [ ]:
def p_overload(n_users, p_active, max_active):
    """P(more than max_active users are active at once) — binomial tail."""
    return sum(math.comb(n_users, k) * p_active**k * (1 - p_active)**(n_users - k)
               for k in range(max_active + 1, n_users + 1))

link, user_rate, p = 1*G, 100*M, 0.10
cap = int(link // user_rate)                    # users the link can carry simultaneously
print('Circuit switching supports', cap, 'users')
pr = p_overload(35, p, cap)
print(f'Packet switching, 35 users: P(11 or more active) = {pr:.5f}')
check('Worked example 3', pr, 0.0004, rel=0.15)

# Monte Carlo: simulate one million instants
rng = np.random.default_rng(1)
active = rng.binomial(35, p, size=1_000_000)
print(f'Simulation: overload in {np.mean(active > cap):.5f} of a million random instants')

In [ ]:
Ns = np.arange(10, 81)
probs = [p_overload(n, p, cap) for n in Ns]
plt.semilogy(Ns, probs, 'o-', ms=3)
plt.axvline(10, color='r', ls='--', label='circuit switching limit (10 users)')
plt.axhline(1e-3, color='gray', ls=':', label='0.1 % of the time')
plt.xlabel('number of users sharing the 1 Gbit/s link'); plt.ylabel('P(demand > capacity)')
plt.title('Statistical multiplexing: how many users can share a link?'); plt.legend(); plt.show()
best = max(n for n, q in zip(Ns, probs) if q < 1e-3)
print(f'Up to {best} users keep overload below 0.1 % — {best/cap:.1f}× what circuit switching admits.')

In [ ]:
# Lecture exercise 3: all 35 users active at once
print(f'P(all 35 active) = 0.1^35 = {0.1**35:.1e}  — practically never.')
print('The relevant risk is "slightly more users than the link carries", i.e. the 0.0004 above.')

## A.5 The four delays and throughput
`d_nodal = d_proc + d_queue + d_trans + d_prop`, with `d_trans = L/R` (depends on packet size and link rate, **not** on distance) and `d_prop = d/s` (depends on distance, **not** on packet size or rate).

In [ ]:
def nodal_delay(L_bits, R_bps, d_m, d_proc=0.0, d_queue=0.0, s=S_FIBRE):
    parts = {'processing': d_proc, 'queueing': d_queue,
             'transmission': t_trans(L_bits, R_bps), 'propagation': t_prop(d_m, s)}
    for k, v in parts.items():
        print(f'  {k:13s} {fmt_time(v)}')
    total = sum(parts.values())
    print(f'  {"TOTAL":13s} {fmt_time(total)}')
    return parts

print('Worked example 4 — Cluj-Napoca → Bucharest, 450 km of fibre, 100 Mbit/s, 1500-byte packet')
p4 = nodal_delay(bits(1500), 100*M, 450e3)
print(f'Propagation / transmission = {p4["propagation"]/p4["transmission"]:.1f}')
print('\nSame packet on a 1 Mbit/s link:')
nodal_delay(bits(1500), 1*M, 450e3);

In [ ]:
# Which one dominates? Transmission vs propagation as the link rate changes (450 km path)
rates = np.logspace(5, 10, 200)         # 100 kbit/s ... 10 Gbit/s
plt.loglog(rates, bits(1500) / rates * 1e3, label='transmission L/R (1500 B)')
plt.loglog(rates, np.full_like(rates, t_prop(450e3) * 1e3), label='propagation d/s (450 km)')
plt.xlabel('link rate (bit/s)'); plt.ylabel('delay (ms)'); plt.legend()
plt.title('The faster the link, the more distance dominates'); plt.show()

In [ ]:
# The convoy analogy from the lecture
cars, booth_s, road_km, speed_kmh = 10, 12, 100, 100
print(f'Transmission (all cars through the booth): {cars*booth_s} s = 2 min')
print(f'Propagation (drive {road_km} km at {speed_kmh} km/h): {road_km/speed_kmh:.0f} h')

# Throughput: the bottleneck link decides
def transfer_time(file_bytes, link_rates_bps):
    bottleneck = min(link_rates_bps)
    return bits(file_bytes) / bottleneck, bottleneck

t, bn = transfer_time(8e6, [100*M, 2*M, 1*G])
print(f'\n8 MB file, bottleneck {bn/M:.0f} Mbit/s → at least {t:.0f} s')

## A.6 Layering and encapsulation in one picture
Each layer adds its header in front of what it receives from above; the receiver removes them in reverse order. Hosts implement 5 layers, routers 3, link-layer switches 2.

In [ ]:
STACK = [  # layer, PDU name, header added (illustrative size in bytes)
    ('Application', 'message',  None),
    ('Transport',   'segment',  ('H_t', 20)),   # e.g. TCP
    ('Network',     'datagram', ('H_n', 20)),   # e.g. IPv4
    ('Link',        'frame',    ('H_l', 18)),   # e.g. Ethernet header + trailer
    ('Physical',    'bits',     None),
]

def encapsulate(message):
    unit, size = [f'[{message}]'], len(message)
    print(f'{"Application":12s} message : {"".join(unit)}   ({size} B)')
    for layer, pdu, hdr in STACK[1:4]:
        unit.insert(0, f'[{hdr[0]}]'); size += hdr[1]
        print(f'{layer:12s} {pdu:8s}: {"".join(unit)}   ({size} B)')
    print(f'{"Physical":12s} {size*8} bits on the wire')
    return size

total = encapsulate('GET /index.html')
DEVICES = {'host': 5, 'router': 3, 'link-layer switch': 2}
print()
for dev, n in DEVICES.items():
    print(f'A {dev:18s} implements layers 1–{n}: {", ".join(l for l, _, _ in STACK[::-1][:n])}')

## A.7 A network of networks
A toy model of Figure 1: tier-1 ISPs peer with each other, regional ISPs buy transit from them, access ISPs buy transit from regional ones, some networks also **peer** at an IXP, and a content provider connects directly to access ISPs. We count how many independently run networks a packet crosses.

In [ ]:
import networkx as nx
G_ = nx.Graph()
customer_provider = [('Access-A', 'Regional-1'), ('Access-B', 'Regional-1'), ('Access-C', 'Regional-2'),
                     ('Access-D', 'Regional-2'), ('Regional-1', 'Tier1-X'), ('Regional-2', 'Tier1-Y'),
                     ('Regional-2', 'Tier1-X'),                     # multi-homing
                     ('Content-CDN', 'Tier1-Y')]
peering = [('Tier1-X', 'Tier1-Y'), ('Access-B', 'Access-C'),        # peering at an IXP
           ('Content-CDN', 'Access-A'), ('Content-CDN', 'Access-D')]
G_.add_edges_from(customer_provider, kind='transit'); G_.add_edges_from(peering, kind='peering')

pos = {'Tier1-X': (-1, 3), 'Tier1-Y': (1, 3), 'Regional-1': (-1.5, 2), 'Regional-2': (1.5, 2),
       'Access-A': (-2.5, 1), 'Access-B': (-0.8, 1), 'Access-C': (0.8, 1), 'Access-D': (2.5, 1), 'Content-CDN': (0, 0)}
plt.figure(figsize=(9, 5))
nx.draw_networkx_nodes(G_, pos, node_color='#cfe3ff', node_size=1800, edgecolors='k')
nx.draw_networkx_labels(G_, pos, font_size=8)
nx.draw_networkx_edges(G_, pos, edgelist=customer_provider, width=2)
nx.draw_networkx_edges(G_, pos, edgelist=peering, style='dashed', width=2, edge_color='tab:orange')
plt.title('Solid: paid customer–provider links · dashed: peering'); plt.axis('off'); plt.show()

for a, z in [('Access-A', 'Access-D'), ('Access-B', 'Access-C'), ('Access-A', 'Content-CDN')]:
    path = nx.shortest_path(G_, a, z)
    print(f'{a} → {z}: {" → ".join(path)}   ({len(path)} networks)')

## A.8 End-of-lecture exercises, checked by code
Try them on paper first, then run the cell.

In [ ]:
print('Exercise 1 — five 1500-byte packets, three 10 Mbit/s links')
t1 = store_and_forward(bits(1500), 10*M, 3, 5)
check('last bit of last packet (s)', t1, 8.4e-3)

print('\nExercise 2 — geostationary satellite, 36 000 km up')
up = 36_000e3 / S_AIR
check('ground → satellite (s)', up, 0.12); check('up and down (s)', 2*up, 0.24)
print('  A question and its answer are ~0.5 s apart: awkward for a phone call.')

print('\nExercise 3 — all 35 users active')
print(f'  0.1^35 = {0.1**35:.0e}')

print('\nExercise 4 — 40 TB from Iași to Bucharest: 100 Mbit/s link vs courier (8 h)')
t_link = bits(40*T) / (100*M)
print(f'  link: {fmt_time(t_link)}   courier: 8 h  →  the courier wins')
check('link time (days)', t_link/86400, 37, rel=0.02)

print('\nExercise 5 — how long is a 12 000-bit packet on 100 Mbit/s fibre?')
length_m = t_trans(12_000, 100*M) * S_FIBRE
check('packet length (km)', length_m/1e3, 24)

**Review questions** (from the lecture notes — answer them in your own words)
1. Host vs packet switch; is a home Wi-Fi router a host? · 2. The three things a protocol defines, in the HTTP example · 3. Why are access links asymmetric? · 4. Why can't a router forward half a packet? · 5. An application better served by circuit switching · 6. Forwarding vs routing · 7. Why do ISPs peer for free? · 8. Which delay changes when 100 Mbit/s → 1 Gbit/s? · 9. Which layers does a router implement?

**✍️ Your answers** (double-click to edit)

- **R1:** 
- **R2:** 
- **R3:** 
- **R4:** 
- **R5:** 
- **R6:** 
- **R7:** 
- **R8:** 
- **R9:** 

---
# Part B · Seminar 1 — Reasoning about networks with numbers
Each problem is first stated briefly; the code computes every answer and checks it against the seminar solution. Change the numbers to explore.

## B.A Warm-up: units

In [ ]:
print('(a) 1500-byte packet           :', bits(1500), 'bits')
print('(b) 5 MB at 20 Mbit/s           :', bits(5*M) / (20*M), 's')
print('(c) distance in fibre in 1 ms   :', S_FIBRE * 1e-3 / 1e3, 'km')
print('(d) "4 MB/s" on a phone         :', 4 * 8, 'Mbit/s')

## B.1 Problem 1 — one link
A ↔ B: R = 10 Mbit/s, 2000 km, s = 2 × 10⁸ m/s, one 1500-byte packet.

In [ ]:
R, d, L = 10*M, 2000e3, bits(1500)
tt, tp = t_trans(L, R), t_prop(d)
check('1. transmission delay (s)', tt, 1.2e-3); check('1. propagation delay (s)', tp, 10e-3)
check('2. last bit arrives at (s)', tt + tp, 11.2e-3)
check('3. first bit is this far from A when A finishes (km)', tt * S_FIBRE / 1e3, 240)
check('4. length where d_trans = d_prop (km)', S_FIBRE * L / R / 1e3, 240)
check('5. length of one bit (m)', S_FIBRE / R, 20)

## B.2 Problem 2 — a path of three links
12 000-bit packet, A → R1 → R2 → B, 20 µs processing per router, empty queues.

| Link | Rate | Length |
|---|---|---|
| A – R1 (campus LAN) | 100 Mbit/s | 1 km |
| R1 – R2 (regional) | 10 Mbit/s | 500 km |
| R2 – B (data centre) | 1 Gbit/s | 50 km |

In [ ]:
links = [('A–R1', 100*M, 1e3), ('R1–R2', 10*M, 500e3), ('R2–B', 1*G, 50e3)]
L, d_proc, n_routers = 12_000, 20e-6, 2
print(f'{"link":6s} {"trans":>10s} {"prop":>10s}')
for name, R_, d_ in links:
    print(f'{name:6s} {fmt_time(t_trans(L, R_)):>10s} {fmt_time(t_prop(d_)):>10s}')
trans = sum(t_trans(L, R_) for _, R_, _ in links)
prop = sum(t_prop(d_) for _, _, d_ in links)
total = trans + prop + n_routers * d_proc
print(f'transmission {fmt_time(trans)} + propagation {fmt_time(prop)} + processing {fmt_time(n_routers*d_proc)}')
check('1. end-to-end delay (s)', total, 4.127e-3, rel=0.005)

bottleneck = min(R_ for _, R_, _ in links)
check('3. max throughput (Mbit/s)', bottleneck / M, 10)
check('3. 30 MB file takes at least (s)', bits(30*M) / bottleneck, 24)
links_up = [('A–R1', 1*G, 1e3)] + links[1:]
print('4. after upgrading A–R1 to 1 Gbit/s the bottleneck is still',
      min(R_ for _, R_, _ in links_up) / M, 'Mbit/s → no speed-up')

## B.3 Problem 3 — why packets? (segmentation and its price)
Message of 1.2 × 10⁶ bits, three 2 Mbit/s links, two store-and-forward switches.

In [ ]:
Msg, R, N = 1.2e6, 2*M, 3
t_msg = store_and_forward(Msg, R, N, 1)
t_pkt = store_and_forward(Msg/100, R, N, 100)
t_hdr = store_and_forward(Msg/100 + 40*8, R, N, 100)
check('1. message switching (s)', t_msg, 1.8)
check('2. 100 packets (s)', t_pkt, 0.612)
check('3. with 40-byte headers (s)', t_hdr, 0.628, rel=0.005)
print(f'   header overhead = {320/12320:.1%}')

# Extension: what packet size minimises the total time?
P = np.arange(1, 2001)
times = (N + P - 1) * (Msg / P + 320) / R
best = P[np.argmin(times)]
plt.plot(P, times, label='with 40-byte header per packet')
plt.plot(P, (N + P - 1) * (Msg / P) / R, '--', label='no header')
plt.axvline(best, color='r', ls=':', label=f'optimum ≈ {best} packets ({Msg/best/8:.0f} B of data each)')
plt.xscale('log'); plt.xlabel('number of packets P'); plt.ylabel('total time (s)'); plt.legend()
plt.title('Too few packets: no pipelining · too many: header overhead'); plt.show()

## B.4 Problem 4 — circuits or packets?
100 Mbit/s link, users send at 10 Mbit/s when active and are active 20 % of the time.

In [ ]:
cap4 = int(100 // 10)
check('1. circuit switching supports (users)', cap4, 10)
check('2. average active users (of 30)', 30 * 0.2, 6)
print('   average load = 60 Mbit/s = 60 % of capacity')
pr4 = p_overload(30, 0.2, cap4)
check('3. P(more than 10 of 30 active)', pr4, 0.026, rel=0.05)

print('\nHomework: users active 50 % of the time, 15 users')
pr_hw = p_overload(15, 0.5, cap4)
print(f'   P(more than 10 of 15 active) = {pr_hw:.3f} → overloaded {pr_hw:.1%} of the time')

for pa, label in [(0.1, '10 % active'), (0.2, '20 % active'), (0.5, '50 % active')]:
    ns = np.arange(10, 61)
    plt.semilogy(ns, [max(p_overload(n, pa, cap4), 1e-12) for n in ns], label=label)
plt.axhline(0.01, color='gray', ls=':'); plt.ylim(1e-8, 1.5)
plt.xlabel('users'); plt.ylabel('P(overload)'); plt.legend()
plt.title('The burstier the users, the more statistical multiplexing gains'); plt.show()

## B.5 Problem 5 — a queue forms
Output link 1 Mbit/s, packets of 12 000 bits → `L/R = 12 ms`. A small simulator of a FIFO queue with a finite buffer reproduces all three cases.

In [ ]:
def fifo(arrivals, L_bits, R_bps, buffer_pkts=None):
    """Simulate a FIFO output queue. Returns per-packet records and the drop times.
    buffer_pkts = packets that may WAIT (the one being transmitted is not counted)."""
    tx = L_bits / R_bps
    in_system = deque()              # departure times of packets still in the router
    records, drops, last_dep = [], [], 0.0
    for a in arrivals:
        while in_system and in_system[0] <= a:
            in_system.popleft()
        waiting = max(len(in_system) - 1, 0)
        if buffer_pkts is not None and in_system and waiting >= buffer_pkts:
            drops.append(a); continue
        start = max(a, last_dep)
        last_dep = start + tx
        in_system.append(last_dep)
        records.append({'arrival': a, 'wait': start - a, 'queue_len': len(in_system) - 1})
    return records, drops

L, R = 12_000, 1*M
r1, _ = fifo([0]*5, L, R)
print('1. five packets at once — waits (ms):', [round(r['wait']*1e3) for r in r1],
      '| average', np.mean([r['wait'] for r in r1])*1e3, 'ms')
r2, _ = fifo(np.arange(200) * 0.012, L, R)
print('2. one every 12 ms — max wait:', round(max(r['wait'] for r in r2)*1e3, 6), 'ms')
arr3 = np.arange(1000) * 0.010
r3, drops = fifo(arr3, L, R, buffer_pkts=20)
print(f'3. one every 10 ms, buffer 20 — first drop at t = {drops[0]:.2f} s; '
      f'afterwards {len(drops)/len(arr3[arr3 >= drops[0]]):.0%} of arrivals are dropped')
for case, gap in [('2', 0.012), ('3', 0.010)]:
    print(f'4. traffic intensity in case {case}: (L/gap)/R = {L/gap/R:.2f}')

t = [r['arrival'] for r in r3]; q = [r['queue_len'] for r in r3]
plt.step(t, q, where='post'); plt.axhline(20, color='r', ls='--', label='buffer = 20 packets')
plt.xlabel('time (s)'); plt.ylabel('packets waiting'); plt.legend()
plt.title('Traffic intensity 1.2: the queue grows until the buffer is full, then packets are lost'); plt.show()

## B.6 Problem 6 — design (and run) a protocol
Below is the seminar's *"one possible answer"* turned into running code: a **time service** over UDP.

* **Format**: ASCII lines ending in CR LF. Request `TIME?`; reply `TIME 2026-10-06T09:30:00Z` (ISO 8601, UTC); otherwise `ERR 400 unknown request`.
* **Order**: client speaks first; exactly one reply per request.
* **Actions**: the client waits 2 s, retries, gives up after 3 tries.

The server can be told to **lose** its first few requests, so you can watch the timeout-and-retry rule work.

In [ ]:
import datetime

class TimeServer(threading.Thread):
    def __init__(self, port=0, drop_first=0):
        super().__init__(daemon=True)
        self.sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        self.sock.bind(('127.0.0.1', port)); self.port = self.sock.getsockname()[1]
        self.drop_first, self.running = drop_first, True
    def run(self):
        while self.running:
            try:
                data, addr = self.sock.recvfrom(1500)
            except OSError:
                break
            if self.drop_first > 0:          # simulate loss of the first requests
                self.drop_first -= 1
                print(f'   [server] request {data!r} LOST in the network'); continue
            line = data.decode('ascii', 'replace')
            if line == 'TIME?\r\n':
                now = datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
                reply = f'TIME {now}\r\n'
            else:
                reply = 'ERR 400 unknown request\r\n'
            self.sock.sendto(reply.encode('ascii'), addr)
    def stop(self):
        self.running = False; self.sock.close()

def time_client(port, request='TIME?', timeout=2.0, tries=3):
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM); s.settimeout(timeout)
    try:
        for attempt in range(1, tries + 1):
            s.sendto((request + '\r\n').encode('ascii'), ('127.0.0.1', port))
            try:
                reply, _ = s.recvfrom(1500)
                print(f'   [client] attempt {attempt}: {reply.decode().strip()}'); return reply
            except socket.timeout:
                print(f'   [client] attempt {attempt}: no reply after {timeout} s')
        print('   [client] giving up → report failure to the user'); return None
    finally:
        s.close()

srv = TimeServer(); srv.start()
print('Correct request:'); time_client(srv.port)
print('Violating the format:'); time_client(srv.port, 'HELLO')
srv.stop()

srv = TimeServer(drop_first=2); srv.start()
print('The first 2 requests are lost (timeout shortened to 0.5 s for the demo):'); time_client(srv.port, timeout=0.5)
srv.stop()
srv = TimeServer(drop_first=5); srv.start()
print('The server is unreachable for a while (5 losses):'); time_client(srv.port, timeout=0.5)
srv.stop()

### Why idempotency matters — the lecture's "transfer 100 lei" question
Asking the time twice is harmless (**idempotent**). A transfer is not. Below, the **reply** is lost, the client retries, and the money moves twice. Adding a **request identifier** lets the server recognise the repetition.

In [ ]:
class Bank:
    def __init__(self):
        self.balance, self.seen = 1000, {}
    def handle(self, msg, use_ids):
        req_id, cmd, amount = msg.split()
        if use_ids and req_id in self.seen:
            return self.seen[req_id]                         # repeated request: same answer, no new transfer
        self.balance -= int(amount)
        reply = f'OK balance={self.balance}'
        self.seen[req_id] = reply
        return reply

def transfer(bank, use_ids, lose_first_reply=True):
    for attempt in (1, 2):
        reply = bank.handle('req-42 TRANSFER 100', use_ids)
        if lose_first_reply and attempt == 1:
            print(f'   attempt 1: server did it ({reply}), but the reply was LOST → client retries')
            continue
        print(f'   attempt {attempt}: {reply}'); break

print('Without request ids:'); b1 = Bank(); transfer(b1, use_ids=False); print('   final balance', b1.balance, '← 200 lei left the account!')
print('With request ids:');    b2 = Bank(); transfer(b2, use_ids=True);  print('   final balance', b2.balance)

## B.7 Problem 7 — the cost of headers (TCP 20 B + IPv4 20 B + Ethernet 14 B header + 4 B trailer)

In [ ]:
HDRS = {'TCP': 20, 'IPv4': 20, 'Ethernet header': 14, 'Ethernet trailer': 4}
for msg in (100, 1460):
    frame = msg + sum(HDRS.values())
    print(f'{msg:5d}-byte message → {frame} byte frame, overhead {sum(HDRS.values())/frame:.1%}')
check('1. frame length', 100 + 58, 158); check('2. frame length', 1460 + 58, 1518)
print('3. switch reads: Ethernet · router: Ethernet + IP · destination host: all headers')

## B.8 Problem 8 — reading a traceroute
The invented output from the seminar sheet is parsed automatically: networks are recognised from the domain names, and the biggest RTT jump is located and turned into a distance.

In [ ]:
TRACE = """
 1  gw.campus.example.edu (10.20.0.1)          0.4 ms   0.3 ms   0.4 ms
 2  core1.campus.example.edu (10.0.0.1)        0.7 ms   0.6 ms   0.7 ms
 3  cta-1.nren.example.net (192.0.2.17)        1.9 ms   1.8 ms   2.1 ms
 4  buc-2.nren.example.net (192.0.2.41)        5.2 ms   5.1 ms   5.4 ms
 5  * * *
 6  vie-1.transit.example.net (198.51.100.9)  22.8 ms  23.1 ms  22.7 ms
 7  fra-3.transit.example.net (198.51.100.33) 34.6 ms  34.2 ms  61.0 ms
 8  nyc-1.transit.example.net (198.51.100.77) 118.3 ms 117.9 ms 118.6 ms
 9  edge.cdn.example.com (203.0.113.5)        117.5 ms 118.1 ms 117.8 ms
"""

def parse_traceroute(text):
    hops = []
    for line in text.strip().splitlines():
        m = re.match(r'\s*(\d+)\s+(.*)', line)
        if not m: continue
        hop, rest = int(m.group(1)), m.group(2)
        name_ip = re.search(r'(\S+)\s+\(([\d.]+)\)', rest)
        ip_only = re.search(r'^\s*([\d.]+)\s', rest)
        name = name_ip.group(1) if name_ip else (ip_only.group(1) if ip_only else '*')
        ip = name_ip.group(2) if name_ip else (ip_only.group(1) if ip_only else None)
        rtts = [float(x) for x in re.findall(r'([\d.]+)\s*ms', rest)]
        hops.append({'hop': hop, 'name': name, 'ip': ip, 'rtts': rtts})
    return hops

def network_of(name):
    parts = name.split('.')
    return '.'.join(parts[-3:-2]) if len(parts) >= 3 else name

hops = parse_traceroute(TRACE)
print(f'{"hop":>3s}  {"router":30s} {"network":10s} {"median RTT":>10s}')
for h in hops:
    med = np.median(h['rtts']) if h['rtts'] else float('nan')
    print(f'{h["hop"]:3d}  {h["name"]:30s} {network_of(h["name"]) if h["ip"] else "-":10s} {med:10.1f}')
nets = []
for h in hops:
    if h['ip'] and (not nets or nets[-1] != network_of(h['name'])):
        nets.append(network_of(h['name']))
print('\n1. Networks crossed:', ' → '.join(nets), f'({len(nets)})')

answered = [h for h in hops if h['rtts']]
jumps = [(b['hop'], a['hop'], min(b['rtts']) - min(a['rtts'])) for a, b in zip(answered, answered[1:])]
to_h, from_h, jump = max(jumps, key=lambda j: j[2])
print(f'2. Largest jump: hop {from_h} → {to_h}, +{jump:.0f} ms RTT → one way ≈ {jump/2:.0f} ms '
      f'→ at most {jump/2*200:.0f} km of fibre (Frankfurt → New York)')
print('3. "* * *": that router does not answer probes; later hops answer, so the path is not broken.')
print('4. 61 ms at hop 7: a momentary queue or a slow reply from the router — one sample in three is noise.')
print('5. Hop 9 < hop 8: possible; different packets at different moments, and routers reply with low priority.')

plt.plot([h['hop'] for h in answered], [np.median(h['rtts']) for h in answered], 'o-')
for h in answered:
    plt.annotate(h['name'].split('.')[0], (h['hop'], np.median(h['rtts'])), fontsize=7,
                 xytext=(3, -10), textcoords='offset points')
plt.xlabel('hop'); plt.ylabel('median RTT (ms)'); plt.title('RTT along the path'); plt.show()

**Homework (Seminar 1):** (1) the lecture exercises without the answers — then compare with A.8; (2) Problem 4 with 50 % activity and 15 users — see B.4; (3) compare **RFC 867** (*Daytime Protocol*) with the design in B.6: what did its author leave unspecified?

**✍️ Your answers** (double-click to edit)

- **Homework 3 — RFC 867 vs our design:** 

---
# Part C · Laboratory 1 (Linux) — exploring the network from the command line
Everything below runs on the Colab VM. **Rules of conduct:** probe only well-known public servers, with the small numbers of packets given here. Do not scan address ranges or ports.

## Task 1 — Who am I?

In [ ]:
sh('hostname')
sh('ip -br link')
sh('ip -br addr')
sh('ip addr show')

In [ ]:
# The same information read directly from the kernel (/sys/class/net)
IFACE, GATEWAY, MY_IP = default_route()
print(f'Interface towards the Internet: {IFACE}   my IPv4: {MY_IP}   default gateway: {GATEWAY}\n')
for ifc in sorted(os.listdir('/sys/class/net')):
    rd = lambda f: open(f'/sys/class/net/{ifc}/{f}').read().strip() if os.path.exists(f'/sys/class/net/{ifc}/{f}') else '?'
    wireless = os.path.exists(f'/sys/class/net/{ifc}/wireless')
    print(f'{ifc:10s} MAC {rd("address"):17s}  MTU {rd("mtu"):5s}  state {rd("operstate"):8s}  {"wireless" if wireless else "wired/virtual"}')
if MY_IP:
    net = ipaddress.ip_interface(re.search(rf'inet ({re.escape(MY_IP)}/\d+)', sh('ip -4 addr', show=False)).group(1))
    print(f'\nMy network: {net.network}  → the first {net.network.prefixlen} bits are shared by every host on my link')

**✍️ Your answers** (double-click to edit)

- **Q1 — interfaces; which one connects you; wired or wireless?:** 
- **Q2 — IPv4/prefix, MAC, MTU; which layer does each address belong to?:** 
- **Q3 — which part is the same as a neighbour’s address? (in Colab: compare with a colleague’s Colab VM):** 

## Task 2 — The way out

In [ ]:
sh('ip route')
sh('ip route get 1.1.1.1')
sh('ip neigh')
sh('cat /etc/resolv.conf')
if have('resolvectl'):
    sh('resolvectl status | head -20')
n_routes = len([l for l in sh('ip route', show=False).splitlines() if l.strip()])
print(f'This host has {n_routes} routes; a core router holds ~1 000 000 (the full IPv4 table).')

**✍️ Your answers** (double-click to edit)

- **Q4 — default gateway; number of routes and why so few are enough:** 
- **Q5 — DNS server; is it in your own network?:** 

## Task 3 — Reachability and round-trip time
`ping` sends ICMP Echo Requests. If the Colab network blocks ICMP you will see 100 % loss — **no reply is weak evidence**. The notebook then measures RTT with a TCP handshake instead (the time of `connect()`).

In [ ]:
UNI_SERVER = 'www.ubbcluj.ro'      # ← put your university's web server here
TARGETS = [GATEWAY or '1.1.1.1', UNI_SERVER, 'www.kernel.org']

def ping_stats(out):
    m = re.search(r'(\d+) packets transmitted, (\d+) received', out)
    r = re.search(r'= ([\d.]+)/([\d.]+)/([\d.]+)/([\d.]+) ms', out)
    ttl = re.search(r'ttl=(\d+)', out)
    return {'sent': int(m.group(1)) if m else 0, 'recv': int(m.group(2)) if m else 0,
            'min': float(r.group(1)) if r else None, 'avg': float(r.group(2)) if r else None,
            'max': float(r.group(3)) if r else None, 'ttl': int(ttl.group(1)) if ttl else None}

results = {}
for t in TARGETS:
    st = ping_stats(sh(f'ping -c 5 -W 2 {t}', timeout=30))
    if st['recv'] == 0 and t != GATEWAY:
        print(f'   ICMP gave no answer → measuring with TCP handshakes instead')
        rt = tcp_rtt(t, 443, 5)
        if rt:
            st.update(min=min(rt), avg=sum(rt)/len(rt), max=max(rt), method='TCP')
    results[t] = st

In [ ]:
print(f'{"destination":22s} {"min":>8s} {"avg":>8s} {"max":>8s} {"loss":>6s} {"TTL":>4s}  method')
for t, st in results.items():
    loss = f'{1 - st["recv"]/st["sent"]:6.0%}' if st['sent'] else '     –'
    f = lambda v: f'{v:8.2f}' if v is not None else '       –'
    print(f'{t:22s} {f(st["min"])} {f(st["avg"])} {f(st["max"])} {loss} {str(st["ttl"] or "–"):>4s}  {st.get("method","ICMP")}')

print('\nQ7 — distance bound from the minimum RTT (light in fibre: 200 km per ms, RTT is there AND back):')
for t, st in results.items():
    if st['min']:
        print(f'   {t:22s} ≤ {st["min"]/2*200:8.0f} km')

print('\nQ9 — routers on the way, estimated from the TTL of the replies:')
for t, st in results.items():
    if st['ttl']:
        start = min(x for x in (64, 128, 255) if x >= st['ttl'])
        print(f'   {t:22s} TTL {st["ttl"]} → probably started at {start} → ~{start - st["ttl"]} routers')

In [ ]:
# Q8 — does packet size change the RTT? (ICMP if possible)
sh('ping -c 5 -W 2 www.kernel.org', timeout=30)
sh('ping -c 5 -W 2 -s 1400 www.kernel.org', timeout=30)

**✍️ Your answers** (double-click to edit)

- **Q6 — table and the dominant delay component:** 
- **Q7 — distance upper bound; why only an upper bound?:** 
- **Q8 — effect of -s 1400; which delay depends on size?:** 
- **Q9 — routers between you and each destination:** 

## Task 4 — The path: a network of networks
Default `traceroute` sends UDP probes; `-I` sends ICMP Echo (like Windows `tracert`); `-T` sends TCP SYN to port 443, which is the probe most often let through by cloud firewalls. Lines of `*` are routers that do not answer.

In [ ]:
DEST = 'www.kernel.org'
out_udp = sh(f'traceroute -n -q 1 -w 2 -m 20 {DEST}', timeout=90)
out_tcp = sh(f'{SUDO}traceroute -T -p 443 -n -q 3 -w 2 -m 20 {DEST}', timeout=120)

In [ ]:
if have('mtr'):
    sh(f'mtr -rwnc 5 {DEST}', timeout=120)

In [ ]:
# Analyse the best trace: names, owners of the address blocks (RDAP), RTT jumps
import urllib.request

def rdap_owner(ip):
    try:
        if ipaddress.ip_address(ip).is_private: return 'private address (this site)'
        with urllib.request.urlopen(f'https://rdap.org/ip/{ip}', timeout=6) as r:
            j = json.load(r)
        return f"{j.get('name','?')} ({j.get('country','?')})"
    except Exception as e:
        return f'? ({type(e).__name__})'

def rdns(ip):
    try: return socket.gethostbyaddr(ip)[0]
    except Exception: return ''

trace = max([parse_traceroute(o) for o in (out_udp, out_tcp)], key=lambda h: sum(1 for x in h if x['ip']))
print(f'{"hop":>3s} {"address":15s} {"RTT ms":>7s}  {"name":40s} owner of the address block')
for h in trace:
    if not h['ip']:
        print(f'{h["hop"]:3d} {"*":15s}'); continue
    print(f'{h["hop"]:3d} {h["ip"]:15s} {min(h["rtts"]) if h["rtts"] else float("nan"):7.2f}  '
          f'{rdns(h["ip"])[:40]:40s} {rdap_owner(h["ip"])}')
ans = [h for h in trace if h['rtts']]
if len(ans) > 1:
    plt.plot([h['hop'] for h in ans], [min(h['rtts']) for h in ans], 'o-')
    plt.xlabel('hop'); plt.ylabel('min RTT (ms)'); plt.title(f'Path to {DEST}'); plt.show()

In [ ]:
# Q13 — a Romanian destination vs one on another continent: where do the paths diverge?
def hop_ips(dest):
    o = sh(f'{SUDO}traceroute -T -p 443 -n -q 1 -w 2 -m 15 {dest}', timeout=90, show=False)
    return [h['ip'] for h in parse_traceroute(o)]
ro, far = hop_ips('www.ubbcluj.ro'), hop_ips('www.usp.br')
common = 0
for a, z in zip(ro, far):
    if a != z: break
    common += 1
print('Romania  :', ro); print('S.America:', far)
print(f'The two paths share their first {common} hops (the same way out of this data centre).')

**✍️ Your answers** (double-click to edit)

- **Q10 — number of hops; agreement with Q9:** 
- **Q11 — the networks crossed (draw them like Figure 1):** 
- **Q12 — biggest RTT increase; which geographic step?:** 
- **Q13 — where the two paths diverge, and why:** 

## Task 5 — Names and addresses (DNS)

In [ ]:
sh('dig www.kernel.org')
sh('dig +short www.kernel.org')
sh('dig +short AAAA www.kernel.org')
router = next((h['ip'] for h in trace if h['ip'] and not ipaddress.ip_address(h['ip']).is_private), '1.1.1.1')
sh(f'dig -x {router} +short')
for i in (1, 2):
    qt = re.search(r'Query time: (\d+) msec', sh('dig www.kernel.org', show=False))
    print(f'query {i}: {qt.group(1) if qt else "?"} ms')

**✍️ Your answers** (double-click to edit)

- **Q14 — addresses (A and AAAA); query time the second time, and why:** 

## Task 6 — A protocol by hand
First watch `curl` speak HTTP (`>` sent, `<` received); then be the browser yourself with `nc` and with a raw Python socket; then break the protocol with `HELLO`.

In [ ]:
sh('curl -sv http://example.org/ -o /dev/null 2>&1 | grep -E "^[<>*]" | head -30')
sh(r"printf 'GET / HTTP/1.1\r\nHost: example.org\r\nConnection: close\r\n\r\n' | nc -w 5 example.org 80 | head -12")

In [ ]:
def raw_http(host, request_lines, port=80):
    """Open a TCP connection and send exactly the lines given, ending each with CR LF."""
    data = b''
    try:
        with socket.create_connection((host, port), timeout=8) as s:
            s.sendall(('\r\n'.join(request_lines) + '\r\n').encode())
            try:
                while chunk := s.recv(4096):
                    data += chunk
            except socket.timeout:
                pass
    except OSError as e:
        print(f'   connection to {host}:{port} failed: {e}')
    return data.decode('latin-1')

good = raw_http('example.org', ['GET / HTTP/1.1', 'Host: example.org', 'Connection: close', ''])
bad = raw_http('example.org', ['HELLO', ''])
print('Correct request →', good.splitlines()[0] if good else '(no reply)')
print('HELLO           →', bad.splitlines()[0] if bad else '(connection closed without a reply)')

In [ ]:
# ss -tn while a connection is open
try:
    s = socket.create_connection(('example.org', 80), timeout=8)
    sh('ss -tn')
    s.close()
except OSError as e:
    print('could not connect:', e); sh('ss -tn')

**✍️ Your answers** (double-click to edit)

- **Q15 — first line of both replies; format, order and actions in your exchange; what did the server do with HELLO?:** 

## Task 7 — Throughput
`TEST_URL` is a test file; replace it with a different one if you want. We repeat the measurement and then run four downloads at once, as if four colleagues downloaded together.

In [ ]:
TEST_URL = 'https://speed.cloudflare.com/__down?bytes=25000000'   # 25 MB test file

def curl_speed(url=TEST_URL, show=True):
    out = sh(f"curl -s -o /dev/null -w '%{{speed_download}} %{{time_total}}' '{url}'", timeout=120, show=False)
    try:
        bps, t = map(float, out.split()[:2])
    except ValueError:
        bps = 0
    if bps == 0:
        print('download failed — check TEST_URL'); return None
    if show: print(f'speed {bps:,.0f} B/s = {bps*8/1e6:7.1f} Mbit/s   time {t:.2f} s')
    return bps * 8 / 1e6

single = [curl_speed() for _ in range(3)]

In [ ]:
res = []
ths = [threading.Thread(target=lambda: res.append(curl_speed(show=False))) for _ in range(4)]
[t.start() for t in ths]; [t.join() for t in ths]
res = [r for r in res if r]
single = [s for s in single if s]
if res and single:
    print('alone          :', ', '.join(f'{x:.0f}' for x in single), 'Mbit/s')
    print('4 in parallel  :', ', '.join(f'{x:.0f}' for x in res), f'Mbit/s  (sum {sum(res):.0f})')
    print('If the sum stays near the single figure, the shared bottleneck is near you (VM uplink);')
    print('if each parallel download is as fast as one alone, the bottleneck was the single flow or the server.')

**✍️ Your answers** (double-click to edit)

- **Q16 — rate in Mbit/s; where is the bottleneck, and what is your evidence?:** 

## Task 8 — Seeing packets (optional)
`tcpdump -e` also prints the link-layer (MAC) header: encapsulation made visible. We capture while generating a little traffic.

In [ ]:
def capture_while(action, bpf, count=8, iface=None, seconds=6):
    iface = iface or IFACE
    p = subprocess.Popen(f'{SUDO}timeout {seconds} tcpdump -i {iface} -n -e -c {count} {bpf}',
                         shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    time.sleep(1.0)
    try:
        action()
    except Exception as e:
        print('action failed:', e)
    print(p.communicate()[0])

capture_while(lambda: sh(f'ping -c 3 -W 1 {GATEWAY or "1.1.1.1"}', show=False), 'icmp')
capture_while(lambda: raw_http('example.org', ['GET / HTTP/1.1', 'Host: example.org', 'Connection: close', '']),
              "'tcp port 80'")

---
# Part D · Laboratory 1 (Windows) — the same ideas, the Windows way
Colab runs Linux, so the PowerShell cmdlets of the Windows sheet (`Get-NetIPConfiguration`, `Test-NetConnection`, …) are not available here. This part gives (1) the **side-by-side table** as a self-test, (2) Python equivalents of the Windows tasks that have no direct Linux twin, and (3) optionally, **real PowerShell 7** on the VM to run the sheet's `TcpClient` script exactly as written.

## D.1 Linux ↔ Windows, from memory
Fill in the dictionary, then run the cell to check yourself (Task 8 of the Windows sheet).

In [ ]:
TABLE = {  # purpose: (Linux, Windows classic, Windows PowerShell)
    'Interfaces and addresses': ('ip addr', 'ipconfig /all', 'Get-NetIPConfiguration'),
    'Forwarding table':        ('ip route', 'route print', 'Get-NetRoute'),
    'Route to one destination': ('ip route get', '–', 'Find-NetRoute'),
    'Neighbours on the link':   ('ip neigh', 'arp -a', 'Get-NetNeighbor'),
    'Reachability, RTT':        ('ping -c', 'ping -n', 'Test-Connection'),
    'Path':                     ('traceroute, mtr', 'tracert, pathping', 'Test-NetConnection -TraceRoute'),
    'Name lookup':              ('dig', 'nslookup', 'Resolve-DnsName'),
    'Test a TCP port':          ('nc -vz host port', '–', 'Test-NetConnection -Port'),
    'Open connections':         ('ss -tn', 'netstat -ano', 'Get-NetTCPConnection'),
    'HTTP client':              ('curl -v', 'curl.exe -v', 'Invoke-WebRequest'),
    'Packet capture':           ('tcpdump, Wireshark', 'pktmon (admin), Wireshark', '–'),
}
MY_ANSWERS = {   # ← type your answers for the Windows classic column
    'Forwarding table': 'route print',
    'Name lookup': '',
    'Open connections': '',
    'Neighbours on the link': '',
}
for k, mine in MY_ANSWERS.items():
    right = TABLE[k][1]
    print(f"{'✅' if mine.strip().lower() == right.lower() else '❌'} {k:26s} you: {mine or '(empty)':15s} answer: {right}")

## D.2 `ipconfig /all` — the same facts gathered in one place
Windows writes the prefix as a **subnet mask**; `255.255.255.0` is `/24`.

In [ ]:
def ipconfig_like():
    out = sh('ip -4 -o addr show', show=False)
    dns = re.findall(r'^nameserver\s+(\S+)', open('/etc/resolv.conf').read(), re.M)
    for m in re.finditer(r'\d+: (\S+)\s+inet (\S+)', out):
        ifc, cidr = m.group(1), ipaddress.ip_interface(m.group(2))
        mac = open(f'/sys/class/net/{ifc}/address').read().strip().replace(':', '-').upper()
        print(f'Adapter {ifc}:')
        print(f'   Physical Address. . . : {mac}')
        print(f'   IPv4 Address. . . . . : {cidr.ip}')
        print(f'   Subnet Mask . . . . . : {cidr.netmask}   (= /{cidr.network.prefixlen})')
        print(f'   Default Gateway . . . : {GATEWAY if ifc == IFACE else ""}')
        print(f'   DNS Servers . . . . . : {", ".join(dns) if ifc == IFACE else ""}\n')
ipconfig_like()
for mask in ['255.255.255.0', '255.255.0.0', '255.255.255.192']:
    print(f'{mask:16s} = /{ipaddress.ip_network("0.0.0.0/" + mask).prefixlen}')

## D.3 `Test-NetConnection -Port` and `Resolve-DnsName`
Port 80 answers at once. Port 81 on example.org is silently dropped, so the client keeps retransmitting its SYN: Windows gives up after about **21 s** (3 + 6 + 12 s), Linux after about **127 s** (6 retries). We cap the wait at 10 s here.

In [ ]:
def test_net_connection(host, port, timeout=10):
    ip = resolve(host)
    if not ip: return
    t0 = time.perf_counter()
    try:
        socket.create_connection((ip, port), timeout=timeout).close(); ok = True
    except Exception as e:
        ok, err = False, type(e).__name__
    dt = time.perf_counter() - t0
    print(f'ComputerName {host}  RemoteAddress {ip}  RemotePort {port}  '
          f'TcpTestSucceeded {ok}  ({dt:.2f} s{"" if ok else ", " + err})')

test_net_connection('example.org', 80)
test_net_connection('example.org', 81)
print('Linux SYN retries on this VM:', open('/proc/sys/net/ipv4/tcp_syn_retries').read().strip())

In [ ]:
def resolve_dnsname(name, family=socket.AF_UNSPEC):
    seen = set()
    try:
        infos = socket.getaddrinfo(name, None, family)
    except OSError as e:
        print(f'{name}: {e}'); return
    for fam, *_, addr in infos:
        if addr[0] not in seen:
            seen.add(addr[0]); print(f'{name:20s} {"AAAA" if fam == socket.AF_INET6 else "A":4s} {addr[0]}')
resolve_dnsname('www.kernel.org')
sh('dig www.kernel.org | grep -A4 "ANSWER SECTION"')   # the TTL column says how long the answer may be cached

## D.4 HTTP by hand — the Windows sheet's `TcpClient` script
Python version first (same steps: open TCP connection → write lines ending in CR LF → read to the end).

In [ ]:
for first_line in ('GET / HTTP/1.1', 'HELLO'):
    reply = raw_http('example.org', [first_line, 'Host: example.org', 'Connection: close', ''])
    print(f'{first_line:15s} → {reply.splitlines()[0] if reply else "(no reply)"}')

**Optional: run the exact PowerShell script.** Set `INSTALL_PWSH = True` (takes about a minute). PowerShell 7 runs on Linux and includes .NET's `System.Net.Sockets.TcpClient`, so the script of the Windows sheet runs unchanged. (Windows-only cmdlets such as `Test-NetConnection` still are not available.)

In [ ]:
INSTALL_PWSH = False

if INSTALL_PWSH and not have('pwsh'):
    ver = re.search(r'VERSION_ID="([\d.]+)"', open('/etc/os-release').read()).group(1)
    sh(f'wget -q https://packages.microsoft.com/config/ubuntu/{ver}/packages-microsoft-prod.deb -O /tmp/ms.deb && '
       f'{SUDO}dpkg -i /tmp/ms.deb > /dev/null && {SUDO}apt-get -qq update > /dev/null && '
       f'{SUDO}apt-get -qq install -y powershell > /dev/null; pwsh -Version', timeout=600)

PS_SCRIPT = r"""
$client = New-Object System.Net.Sockets.TcpClient('example.org', 80)
$stream = $client.GetStream()
$writer = New-Object System.IO.StreamWriter($stream)
$writer.NewLine = "`r`n"; $writer.AutoFlush = $true
$reader = New-Object System.IO.StreamReader($stream)
$writer.WriteLine('GET / HTTP/1.1')
$writer.WriteLine('Host: example.org')
$writer.WriteLine('Connection: close')
$writer.WriteLine('')
$reader.ReadToEnd().Split("`n")[0..8] -join "`n"
$client.Close()
"""
if have('pwsh'):
    open('/tmp/http.ps1', 'w').write(PS_SCRIPT)
    sh('pwsh -NoProfile -File /tmp/http.ps1', timeout=60)
else:
    print('PowerShell is not installed (set INSTALL_PWSH = True to try it).')

## D.5 Questions specific to the Windows sheet
* **Q3 (Windows)** — Was the address configured by DHCP? On Linux: `ip addr` shows `dynamic` and the remaining lifetime (`valid_lft`).
* **Q9 (Windows)** — Windows Firewall does not answer ping from other machines by default. What does a failed ping prove?
* **Q10 (Windows)** — Initial TTLs: Linux and network equipment 64 or 255, Windows 128.
* **Q13 (Windows)** — `pathping` shows loss at an intermediate hop but not at the destination: is there a real problem?

In [ ]:
sh(f'ip addr show {IFACE} | grep -E "inet |valid_lft"')
print('Initial TTL on this Linux VM:', open('/proc/sys/net/ipv4/ip_default_ttl').read().strip())

**✍️ Your answers** (double-click to edit)

- **Q3 (Windows) — DHCP?:** 
- **Q9 (Windows) — what a failed ping proves:** 
- **Q13 (Windows) — loss only at an intermediate hop:** 

---
### Next week
Lecture 2: layered architectures in detail, OSI vs TCP/IP, and a quantitative treatment of delay, loss and throughput. Laboratory 2: packet capture — see the **Week 2** notebook.